# Imports

In [1]:
import pandas as pd

# Load data

In [2]:
df = pd.read_parquet('wave31_subset_cols.parquet')

# Add net contribution columns

In [3]:
to_add = []

for c in ['al2_net1', 'lr1_net1', 'al2_net2', 'lr1_net2']:
    s = (df[c] * df['wt_new_']).rename(f'{c}_wt')
    to_add.append(s)

In [4]:
df = pd.concat([df, *to_add], axis='columns')

# Function definitions

In [5]:
def calc_effective_n(ser):
    return (ser.sum() ** 2) / (ser ** 2).sum()

In [6]:
sample_size_map = {
    'count': 'Unweighted',
    'sum': 'Weighted',
    'calc_effective_n': 'Effective',
}

def calc_net_support(df, grouper, net_score='net1'):
    if type(grouper) is str:
        grouper = [grouper]
    plot_df = df.groupby(grouper)[[f'al2_{net_score}_wt', f'lr1_{net_score}_wt']].sum().div(df.groupby(grouper)['wt_new_'].sum(), axis=0) * 100
    effective_n = df.groupby([*grouper])['wt_new_'].agg(['count', calc_effective_n, 'sum'])
    name_map = {f'al2_{net_score}_wt': 'Pro Death Penalty', f'lr1_{net_score}_wt': 'Pro Wealth Redistribution'}
    return plot_df.join(effective_n).rename(columns={**sample_size_map, **name_map})

# Plots

### Party only

In [7]:
party_map = {
    'CON': 'Conservative',
    'DNK': "Don't know",
    'GRN': 'Green',
    'LAB': 'Labour',
    'LDM': 'Liberal Democrat',
    'OTH': 'Other parties',
    'PLC': 'Plaid Cymru',
    'RFM': 'Reform UK',
    'SNP': 'SNP',
    'WNV': 'Would not vote',
}

In [8]:
plot_df = calc_net_support(df, ['party_code']).reset_index()
plot_df['party_name'] = plot_df['party_code'].map(party_map)
plot_df.to_csv('plot_data/party_bubble.csv', index=False)

### Age and gender

In [9]:
plot_df = calc_net_support(df, ['age1', 'gendername']).reset_index()
plot_df.to_csv('plot_data/age_gender_bubble.csv', index=False)

### Religion granular

In [10]:
plot_df = calc_net_support(df, ['rel1']).drop('Unanswered').reset_index()
plot_df = plot_df[plot_df['Unweighted'] > 75]
plot_df.to_csv('plot_data/religion_granular_bubble.csv', index=False)

### Party or religion

In [11]:
main_parties = ['CON', 'LAB', 'PLC', 'SNP', 'LDM', 'RFM', 'GRN']

In [12]:
a = calc_net_support(df, ['party_code']).loc[main_parties].reset_index().rename(columns={'party_code': 'colour'})
a['label'] = a['colour'].map(party_map)

In [13]:
b = calc_net_support(df, ['rel2']).drop('Unanswered').reset_index().rename(columns={'rel2': 'colour'})
b['label'] = b['colour']

In [14]:
pd.concat([a,b]).to_csv('plot_data/party_or_religion_bubble.csv', index=False)

### Party and religion combined

In [15]:
five_parties = ['CON', 'LAB', 'LDM', 'RFM', 'GRN']

In [16]:
plot_df = calc_net_support(df, ['party_code', 'rel3']).loc[five_parties].reset_index()
plot_df = plot_df[~(plot_df['rel3'] == 'Unanswered')]
plot_df['party_name'] = plot_df['party_code'].map(party_map)
plot_df.to_csv('plot_data/party_x_religion_bubble.csv', index=False)